# WildSpeech M8.1 — GitHub Edition: Score-Aware Discovery, Acoustic Intelligence & Experimental Atlas

**End-to-end research workflow:** official-data audit → lightweight acoustics → optional GPU phonetic CTC → M8 models → fold-matched modality ablations and permutation controls → double cross-fitted conditional associations → cluster stability and confounding → school bootstrap → figure atlas and evidence report → competition-format submission.

**Evidence hierarchy:** outer CV and independent transfer checks support predictive claims; training-only group signatures and feature–score correlations support hypotheses only. Do not interpret acoustic quality proxies as speaker diarization.

## 0. Environment and paths

In [ ]:
# Reproducible GitHub-first Colab setup. No ZIP upload is required.
from pathlib import Path
import os, sys, subprocess
REPO_URL = "https://github.com/mybenkhadda/WildSpeech.git"
REPO_REF = "main"
REPO_ROOT = Path('/content/WildSpeech_git') if Path('/content').exists() else Path.cwd()
if not (REPO_ROOT/'WildSpeech_M8'/'pyproject.toml').is_file():
    if Path('/content').exists():
        subprocess.check_call(['git','clone','--depth','1','--branch',REPO_REF,REPO_URL,str(REPO_ROOT)])
    elif (Path.cwd()/'WildSpeech_M8'/'pyproject.toml').is_file():
        REPO_ROOT = Path.cwd()
    else:
        raise FileNotFoundError('Run from a WildSpeech git checkout, or use Colab to auto-clone the repo')
PROJECT_ROOT = REPO_ROOT/'WildSpeech_M8'
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Local environment; Google Drive mount skipped')
DATA_ROOT = Path(os.environ.get('WILDSPEECH_DATA_ROOT','/content/drive/MyDrive/WildSpeech'))
OUTPUT_ROOT = Path(os.environ.get('M8_OUTPUT_ROOT','/content/drive/MyDrive/WildSpeech_M8'))
PROFILE_CSV = os.environ.get('M71_PROFILE_CSV','')
ASR_PATH = os.environ.get('M8_ASR_PATH','')
MODE = 'FULL'  # use 'SMOKE' to validate paths on a few clips
RUN_AUDIO = True
RUN_SPEECH = bool(ASR_PATH)
RUN_PHONETIC = False  # GPU optional; verify model license and validity before enabling
PHONETIC_BATCH = 3
RUN_RESEARCH = True
print({'repo':str(REPO_ROOT),'package':str(PROJECT_ROOT),'data':str(DATA_ROOT),
       'output':str(OUTPUT_ROOT),'mode':MODE})


## 1. Install requirements and execute tests

In [ ]:
assert (PROJECT_ROOT/'pyproject.toml').is_file(), 'Missing WildSpeech_M8/pyproject.toml'
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',str(PROJECT_ROOT)+'[test]'])
subprocess.check_call([sys.executable,'-m','pytest','-q',str(PROJECT_ROOT/'tests'/'test_m8.py'),
                       str(PROJECT_ROOT/'tests'/'test_m81.py')])


## 2. Official input and provenance validation

In [ ]:

from wildspeech_m8.m8_pipeline import Config,read_official,discover_profile,feature_families
config=Config(data_root=str(DATA_ROOT),output_root=str(OUTPUT_ROOT),profile_csv=PROFILE_CSV,
              quick=(MODE=='SMOKE'),loso=(MODE!='SMOKE'))
tr_meta,te_meta,labels,submission_schema,clips=read_official(config)
resolved_profile=discover_profile(config)
print('TRAIN',len(tr_meta),'TEST',len(te_meta),'CLIPS', {k:len(v) for k,v in clips.items()})
print('M7 profile:',resolved_profile if resolved_profile else 'MISSING — limited metadata fallback')
print('Tasks:',clips['train']['task'].value_counts().to_dict())
print('Train schools:',tr_meta['school_id'].value_counts().to_dict() if 'school_id' in tr_meta else 'Not provided')
print('Target summary:',labels['outcome_score'].describe())
assert set(labels.subject_id)==set(tr_meta.subject_id)
assert set(submission_schema.subject_id)==set(te_meta.subject_id)

## 3. Acoustic extraction and task-aware aggregation

The `m8_audio.py` extractor **streams the tar files once**, caches clip-level features, aggregates by student and task, logs errors, and shows progress bars. `SMOKE` extracts only 50 clips per split; `FULL` extracts all. These features are robust signal statistics and VAD proxies, **not validated speaker identity or phonetic labels**.

In [ ]:
if RUN_AUDIO:
    command=[sys.executable,'-m','wildspeech_m8.m8_audio','--data-root',str(DATA_ROOT),
             '--output-root',str(OUTPUT_ROOT),'--limit',str(50 if MODE=='SMOKE' else 0)]
    subprocess.check_call(command)
else:
    print('Skipping new audio extraction; cached audio_student_features.csv will be used if present')

## 4. Optional reference-conditioned ASR diagnostics

Provide a previously generated, locally obtained ASR export with `filename`, `transcript` (or `asr_transcript`) and `split` if filenames overlap. Similarity to expected text measures ASR output agreement, not verified child articulation. Avoid sending private audio to external APIs.

In [ ]:
if RUN_SPEECH:
    assert Path(ASR_PATH).is_file(),f'Missing local ASR file: {ASR_PATH}'
    subprocess.check_call([sys.executable,'-m','wildspeech_m8.m8_speech','--data-root',str(DATA_ROOT),
                           '--asr-path',ASR_PATH,'--output-root',str(OUTPUT_ROOT)])
else:
    print('ASR not supplied; phonetic/transcript diagnostics are omitted rather than invented')

## 4B. Optional local Pasketti-phonetic inference (GPU)

Uses the officially demonstrated `drivendata/pasketti-phonetic` WavLM CTC model. Installs optional dependencies and **runs only when enabled**. Processes audio locally from tar archives in batches, caches outputs, aligns hypotheses with expected IPA using `eng_to_ipa`, and creates task-level student features. Model hypotheses and dictionary reference distances are **not gold child pronunciations** and are vulnerable to adult-voice contamination and anonymization artifacts.

In [ ]:
if RUN_PHONETIC:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(PROJECT_ROOT/'optional-requirements.txt')])
    cmd=[sys.executable,'-m','wildspeech_m8.m8_phonetic','--data-root',str(DATA_ROOT),
         '--output-root',str(OUTPUT_ROOT),'--batch-size',str(PHONETIC_BATCH)]
    if MODE=='SMOKE':cmd+=['--limit','25']
    subprocess.check_call(cmd)
else:
    print('Pasketti phonetics off; enable RUN_PHONETIC=True for local pretrained phonetic inference.')

## 5. Run the full discovery pipeline

Includes a global CatBoost baseline; 2/3-group supervised latent mixtures; residual-rule discovery; **cross-fitted** local residual targets; nested candidate selection; paired bootstrap; school-transfer evaluation; original and transformed features; full research figures and the submission CSV. CPU is adequate for the tabular models. GPU is only potentially useful for separate pretrained speech encoder extraction.

In [ ]:
from wildspeech_m8.m8_pipeline import run
config=Config(data_root=str(DATA_ROOT),profile_csv=str(resolved_profile) if resolved_profile else '',
              output_root=str(OUTPUT_ROOT),quick=(MODE=='SMOKE'),loso=(MODE!='SMOKE'),
              cv_folds=5,cv_repeats=2,inner_folds=3,bootstrap_n=400,
              max_features=450)
result_dir=run(config)

## 5B. M8.1 — independent feature-family experiments and discovery atlas

**New experiments:** same-fold screening/context controls; acoustic/speech/temporal ablations; within-fold shuffled-modality negative controls; double cross-fitted conditional associations; FDR correction; student- and school-cluster bootstrap; descriptive group perturbation stability; train/test feature shift; matched students; instrument bias. The research atlas is saved as `RESEARCH_ATLAS.html` plus underlying CSVs and PNGs. These experiments do not automatically select the submission model.

In [ ]:
if RUN_RESEARCH:
    from wildspeech_m8.m8_research import ResearchConfig,run_research
    exp_cfg=ResearchConfig(folds=5,repeats=2,permutations=3,bootstrap=800,
                           stability_resamples=25,max_assoc_features=350,
                           seed=config.seed+1,quick=(MODE=='SMOKE'))
    exp_result=run_research(config,exp_cfg)
    print(exp_result)
else:
    print('Enhanced experiments skipped. RUN_RESEARCH=True is recommended.')

## 5C. Incremental speech and acoustic evidence

Study the feature-family gains with the same student splits, not isolated single-feature p-values. A stable, positive increment beyond screening/context is more compelling than visually separated clusters. Bootstrap intervals and shuffled speech controls quantify uncertainty and potential artifacts.

In [ ]:
from IPython.display import display,Image
import pandas as pd
TABLES=OUTPUT_ROOT/'tables';FIGS=OUTPUT_ROOT/'figures'
for name in ['ablation_summary.csv','paired_student_bootstrap_ablation.csv','paired_school_bootstrap_ablation.csv']:
    f=TABLES/name
    if f.is_file():
        print('\n',name)
        display(pd.read_csv(f).round(5).head(30))
for name in ['E01_feature_family_ablation.png','E02_fold_gain.png','E05_student_bootstrap.png',
             'E06_school_bootstrap.png']:
    f=FIGS/name
    if f.is_file():display(Image(filename=str(f)))

## 5D. Conditional feature–score relationships

These results use OOF screening residuals and OOF feature residuals, then report partial rank association and BH-adjusted q-values. This is a **screening-adjusted descriptive association**, not proof that a feature causes a reading outcome or that it improves prediction. Compare against the paired ablation tables.

In [ ]:
f=TABLES/'conditional_feature_score_associations.csv'
if f.is_file():
    t=pd.read_csv(f)
    display(t[['feature','family','n','raw_rho','conditional_rho','conditional_p_unadjusted','q_bh']].head(30))
    print('BH q<0.10:',int((t.q_bh<.1).sum()),'out of',len(t),'exploratory comparisons')
for name in ['E03_conditional_associations.png','E04_association_fdr.png']:
    f=FIGS/name
    if f.is_file():display(Image(filename=str(f)))

## 5E. Cluster robustness, similar students with divergent outcomes, and covariate shift

A high ARI demonstrates stable assignments under subset perturbations; it does **not** prove distinct phenotypes. Nearest-neighbor score gaps are descriptive, and train/test KS statistics are unsupervised drift alarms rather than outcome guarantees.

In [ ]:
for name in ['descriptive_group_stability.csv','descriptive_group_summary.csv',
             'ability_matched_pairs.csv','feature_distribution_shift.csv','instrument_oof.csv']:
    f=TABLES/name
    if f.is_file():
        print('\n',name)
        display(pd.read_csv(f).head(20))
for name in ['E07_train_test_shift.png','E08_group_stability.png','E09_group_residuals.png',
             'E10_matched_heterogeneity.png','E11_instrument_bias.png','E12_residual_vs_prediction.png','E13_loso_school_deltas.png']:
    f=FIGS/name
    if f.is_file():display(Image(filename=str(f)))

## 5F. Reproducible evidence report

The report includes the observed metrics, conditional associations, uncertainty intervals, stability diagnostics and interpretation caveats. **Results in this report are from the current run**; the notebook ships without official-data scores.

In [ ]:
from IPython.display import Markdown
r=OUTPUT_ROOT/'RESEARCH_SUMMARY.md'
if r.is_file():display(Markdown(r.read_text()))
print('Complete figure index:',OUTPUT_ROOT/'RESEARCH_ATLAS.html')
print('Enhanced figures:',len(list((OUTPUT_ROOT/'figures').glob('E*.png'))))

## 6. Compare held-out student CV methods

In [ ]:
import pandas as pd,numpy as np,matplotlib.pyplot as plt
from IPython.display import display,Image
TABLES=OUTPUT_ROOT/'tables';FIGS=OUTPUT_ROOT/'figures'
metrics=pd.read_csv(TABLES/'cv_metrics.csv')
if metrics.empty:print('No complete CV repeat in SMOKE mode. Run FULL for valid CV estimates.')
else:
    display(metrics.groupby('model')[['r2','mae','rmse']].agg(['mean','std']).round(5))
    display(Image(filename=str(FIGS/'06_model_comparison.png')))
    bt=TABLES/'paired_bootstrap.csv'
    if bt.is_file():display(pd.read_csv(bt));display(Image(filename=str(FIGS/'07_paired_bootstrap.png')))

## 7. Feature–score exploratory atlas

Univariate associations are **descriptive and multiply tested**. Use the held-out incremental evaluation to determine predictive value. Figures `12_*` show highest absolute Spearman associations with response-binned overlays.

In [ ]:
assoc=TABLES/'exploratory_univariate_associations.csv'
if assoc.exists():
    display(pd.read_csv(assoc).head(25))
    display(Image(filename=str(FIGS/'11_feature_score_associations.png')))
    for fig in sorted(FIGS.glob('12_feature_response_*.png')):
        display(Image(filename=str(fig)))
else:print('Feature atlas not yet generated')

## 8. Outcome residuals and group signatures

The descriptive fit uses all labeled training students, so it helps develop hypotheses but **cannot establish held-out subgroup discovery**. Interpret assignment softmax scores as mixture diagnostics, not calibrated scientific membership probabilities.

In [ ]:
for name in ['04_oof_calibration.png','05_oof_residuals.png','08_group_residuals.png','09_group_assignment.png','group_feature_signatures.png']:
    p=FIGS/name
    if p.exists():display(Image(filename=str(p)))
for fn in ['descriptive_group_assignments.csv','group_signature_zscores.csv','permutation_group_diagnostics.csv']:
    p=TABLES/fn
    if p.exists():print(fn);display(pd.read_csv(p).head(12))

## 9. School-transfer robustness

In [ ]:
p=TABLES/'loso_per_school.csv'
if p.exists():
    school=pd.read_csv(p)
    display(school.groupby('model').agg(student_count=('n','sum'),mean_school_mae=('mae','mean'),mean_school_rmse=('rmse','mean')).round(4))
    display(school.sort_values(['school','model']).head(40))
    q=FIGS/'10_loso_per_school.png'
    if q.exists():display(Image(filename=str(q)))
else:print('LOSO skipped (SMOKE or insufficient schools).')

## 10. Acoustic feature coverage and quality

Check invalid clips, task-specific distribution shifts, and per-student missingness. Audio signal features may encode recording conditions and anonymization, so downstream predictions require controls.

In [ ]:
for split in ['train','test']:
    p=TABLES/f'audio_clip_features_{split}.csv'
    if not p.exists():continue
    d=pd.read_csv(p)
    print(split,'clips=',len(d),'usable_proxy_rate=',d.get('audio__quality_usable',pd.Series(dtype=float)).mean())
    if 'task' in d and 'audio__onset_s_proxy' in d:
        display(d.groupby('task').agg(n=('filename','size'),onset=('audio__onset_s_proxy','median'),
                speech_fraction=('audio__speech_fraction_proxy','median'),quality=('audio__quality_usable','mean')))
        fig,ax=plt.subplots(figsize=(8,4))
        d.boxplot(column='audio__onset_s_proxy',by='task',ax=ax,showfliers=False)
        plt.suptitle('');plt.title(f'{split}: onset proxy by task');plt.ylabel('Seconds');plt.xticks(rotation=20)
        plt.show()

## 11. Final model and prediction export

The final candidate is selected using training-only inner CV. The file is generated only in FULL mode. Inspect official submission schema before upload.

In [ ]:
submission=OUTPUT_ROOT/'predictions'/'m8_submission.csv'
if submission.exists():
    out=pd.read_csv(submission,dtype={'subject_id':str});display(out.head())
    print('Rows=',len(out),'Columns=',out.columns.tolist(),'Score range=',(out.outcome_score.min(),out.outcome_score.max()))
    assert len(out)==len(submission_schema) and list(out.subject_id)==list(submission_schema.subject_id)
    assert out.outcome_score.notna().all()
else:print('No final submission in SMOKE mode. Run FULL.')

## 12. Interpretation checklist

Before claiming a new literacy phenotype or submitting a model:

1. Verify **all** relevant input families are actually present and not systematically missing (`feature_manifest.csv`).
2. Treat **nested-selected** held-out results as the meaningful model-selection estimate; avoid choosing a model simply because it appears best on the final outer folds.
3. Compare the global baseline to residual grouping using R²/MAE/RMSE, paired uncertainty and per-school metrics.
4. Interpret groups by ability-adjusted acoustic/behavioral signatures. Re-evaluate group stability and instruments before making substantive claims.
5. Audit correlations with school, item difficulty, audio quality, and presence of adult speech. The supplied lightweight extraction does **not** perform validated diarization.
6. Do **not** treat ASR errors as confirmed child errors. Add appropriately licensed local child-ASR and validated forced alignment as a separate model-backed extension after checking accuracy on permitted development data.
7. Run the full official dataset (no clip caps), freeze configuration, archive software/weight versions, then submit the official-format CSV.

**If the local group correction underperforms, retain the global model.** Discovery and prediction are related but distinct objectives.